# Run Veil in Colab
Choose a T4 GPU under Runtime → Change runtime type. Run one cell at a time. This is a supervised experimental demo, not production hosting.


In [ ]:
from google.colab import files
print('Select Veil_GitHub_Folders.zip')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Choose exactly one ZIP file.')


In [ ]:
import zipfile
from pathlib import Path
with zipfile.ZipFile(next(iter(uploaded))) as z:
    z.extractall('/content')
%cd /content/veil_mvp
print('Project files:', [p.name for p in Path('.').iterdir()])


In [ ]:
# Use Colab's current GPU libraries; these package versions need testing on your runtime.
%pip install -r requirements-colab.txt


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
if not torch.cuda.is_available():
    raise RuntimeError('Select a T4 GPU before continuing.')


## Start the server
The access key and temporary link are private. Do not publish notebook output containing them.


In [ ]:
import os, secrets, subprocess, time, urllib.request
from pathlib import Path
os.environ['VEIL_API_TOKEN'] = secrets.token_urlsafe(24)
server_log = open('/content/veil-server.log', 'w')
server = subprocess.Popen(['python', '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '8000'], cwd='/content/veil_mvp', stdout=server_log, stderr=subprocess.STDOUT)
for i in range(60):
    time.sleep(1)
    try:
        print('Health:', urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2).read().decode())
        break
    except Exception:
        if server.poll() is not None:
            print('Server stopped. Log:', Path('/content/veil-server.log').read_text())
            raise RuntimeError('Server did not start')
else:
    raise RuntimeError('Server timed out. Read /content/veil-server.log')
print('PRIVATE ACCESS KEY:', os.environ['VEIL_API_TOKEN'])


## Make a temporary website link
This exposes a private demo server to the internet. Keep both the link and key private. This cell downloads cloudflared from GitHub; its executable is not pinned or verified in this prototype.


In [ ]:
import re
from pathlib import Path
result = subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/content/cloudflared'])
if result.returncode:
    raise RuntimeError('Tunnel download failed')
Path('/content/cloudflared').chmod(0o755)
tunnel_log = open('/content/veil-tunnel.log', 'w')
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stdout=tunnel_log, stderr=subprocess.STDOUT)
url = None
for i in range(60):
    time.sleep(1)
    text = Path('/content/veil-tunnel.log').read_text()
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if match:
        url = match.group(0)
        break
if not url:
    print(text[-3000:])
    raise RuntimeError('No link appeared')
print('PRIVATE WEBSITE LINK:', url)


Open the link, enter the key, and upload a non-sensitive photo. The first run downloads model files and can take several minutes. The website and results are experimental.


In [ ]:
# Run when finished; the link will stop working.
for process in (globals().get('tunnel'), globals().get('server')):
    if process and process.poll() is None:
        process.terminate()
print('Stop requested.')
